<h1 align="center">Laboratorio 10</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/labs/lab10)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert lab10.ipynb --to html

## Task 1

En clase explicamos que la Difusión "esculpe" la imagen restando ruido de un tensor dentro del Espacio Latente, paso a paso, antes de que el Decoder la convierta en píxeles. Usted deberá demostrar este comportamiento interrumpiendo el proceso para observar la evolución temporal.

Para esto considere las siguientes instrucciones paso a paso:

1. Instancie un modelo estándar de difusión (por ejemplo, StableDiffusionPipeline usando los pesos de la versión 1.5). Asegúrese de enviar el modelo a la GPU (cuda).
2. Defina exactamente 20 pasos de inferencia (num_inference_steps=20) y fije una semilla aleatoria (Seed) usando torch.manual_seed() para garantizar la reproducibilidad.
3. Utilice un prompt que exija al modelo generar estructuras geométricas y texturas complejas. Ejemplo obligatorio: "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution".
4. Usted no debe generar la imagen final de un solo golpe. Su código debe interceptar el tensor latente (matriz matemática de ruido) durante el loop de denoising.
   1. Hint: Para lograr esto sin escribir el pipeline desde cero, investigue en la documentación de Hugging Face el uso de la función callback_on_step_end dentro de la llamada al pipeline, o bien, desempaquete el pipeline y escriba el ciclo for manualmente usando el scheduler.step().
5. Guarde una copia del tensor latente en memoria exactamente en los pasos 4, 10, 16 y 20.
6. Tome esos 4 tensores latentes y páselos manualmente por el componente decodificador de su modelo (vae.decode()).
   1. Warning: Recuerde que los modelos latentes escalan matemáticamente los tensores. Antes de pasar su tensor al VAE, investigue si debe multiplicarlo o dividirlo por el scaling_factor del VAE para evitar errores de dimensión o imágenes en negro.
7. Convierta la salida del VAE a formato de imagen (PIL) y guárdelas.

De esta parte se espera que entregue y responda:

- Muestre la cuadrícula visual con la evolución de las 4 imágenes.
- Observe la diferencia entre la imagen del paso 4 y la del paso 16. Explicado con base en la teoría de frecuencias espaciales y Cross-Attention: ¿Qué características de la imagen (forma global, colores bases, silueta vs. brillos, texturas finas, detalles del neón) resuelve la U-Net en las etapas iniciales de ruido alto, y qué resuelve en las etapas finales de ruido bajo? Justifique técnicamente su respuesta.


## Task 2

Como vimos al final de la sesión, el modelo Nano Banana de Google representa la vanguardia en eficiencia: usar técnicas de destilación para saltarse pasos matemáticos de la Cadena de Markov y correr modelos generativos en milisegundos. Usted simulará este escenario midiendo empíricamente el trade-off (costo-beneficio) en producción.

Para esto considere las siguientes instrucciones paso a paso:

1. Escenario A (Modelo Estándar - Costoso):

   1. Utilice el modelo estándar de la Sección 1.
   2. Configure la generación a 50 pasos de inferencia.
   3. Use la misma semilla y el mismo prompt.
   4. Implemente medidores de rendimiento en su código: use time.time() para medir los segundos exactos que tarda la inferencia, y torch.cuda.max_memory_allocated() para medir el pico de VRAM consumida.

2. Escenario B (Modelo Destilado - Eficiente):

   1. Para simular el paradigma de "Nano Banana", usted debe cargar una arquitectura diseñada para pocos pasos (Destilación).
   2. Investigue e instancie un modelo basado en Turbo o LCM. (Ejemplo: SDXL-Turbo, SD-Turbo, o cargar pesos LCM en su modelo base).
   3. Configure la generación a únicamente 4 pasos de inferencia.
   4. Mida y registre el tiempo de ejecución y el consumo máximo de VRAM de la misma manera que en el Escenario A.

De esta parte se espera que entregue y responda:

- Presente las dos imágenes resultantes lado a lado.
- Muestre una tabla comparando: Modelo usado, Pasos, Tiempo de Ejecución (segundos) y VRAM (MB/GB).
- Como Arquitecto de IA encargado de desplegar esta API en una aplicación con millones de usuarios:
  - Describa brevemente cómo la técnica de "Destilación" permitió al modelo B generar una imagen coherente en solo 4 pasos, mientras que si usted pusiera el modelo A a 4 pasos el resultado sería ruido inservible.
  - Analice sus métricas (Tiempo y Memoria). Con base en la calidad visual obtenida frente al ahorro de hardware, emita un dictamen justificando cuál de los dos escenarios elegiría para producción y por qué.

Recuerden: Las respuestas vagas no sumarán puntos. Se busca que demuestre dominio técnico del flujo de datos (Tensores → U-Net → VAE → Pixeles) y capacidad de toma de decisiones arquitectónicas en ecosistemas de Inteligencia Artificial.